In [ ]:
pip install "gymnasium[classic-control]" stable-baselines3 sb3-contrib pyyaml pydantic

In [ ]:
pip uninstall -y torch torchvision torchaudio torchdata torchtext dgl

In [ ]:
pip install torch==2.1.2 

In [ ]:
pip install torchdata==0.7.1

In [ ]:
pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html

In [ ]:
pip install "numpy<2.0.0"

In [ ]:
pip install holidays

In [13]:
import datetime
import json
import numpy as np
import pandas as pd
import gymnasium as gym
from pathlib import Path

DEMAND = 0
CAPACITY = 1

# Potential-based shaping weights (Ng, Harada & Russell 1999): roughly what one
# unit costs under the step + terminal rewards.
SHAPE_W_MIN = 16.0       # uncoverable minimum: -5 skip, -1 lost fill, -10 terminal
SHAPE_W_STRANDED = 1.8   # stranded workday: overfill +0.2 instead of +1, -1 terminal

def _build_special_days(year, n_days):
    import holidays as hl
    pt_holidays = hl.country_holidays("PT", years=[year])
    start = datetime.date(year, 1, 1)
    special = set()
    for d in range(n_days):
        date = start + datetime.timedelta(days=d)
        if date.weekday() == 6 or date in pt_holidays:
            special.add(d)
    return special


class ScheduleEnv(gym.Env):
    def __init__(self,
                 data_dir: str = "../../../../data/problems/SMARTASK_SIMPLE_2025",
                 capacity_slack: int = 1):
        super().__init__()
        base = Path(data_dir)

        with open(base / "problem.json") as f:
            prob = json.load(f)

        self.num_days = prob["temporalScope"]["numDays"]
        self.year = prob["temporalScope"]["year"]
        employees = prob["employees"]["simple"]
        self.num_employees = len(employees)
        self.employee_teams = [set(emp.get("teams", [])) for emp in employees]
        self.dual_team = [len(teams) > 1 for teams in self.employee_teams]

        shifts_sorted = sorted(prob["demand"]["shifts"], key=lambda s: s["order"])
        self.shift_codes = [s["code"] for s in shifts_sorted]
        self.shift_idx = {c: i for i, c in enumerate(self.shift_codes)}
        self.shift_order_map = {s["code"]: s["order"] for s in shifts_sorted}
        self.teams = list(prob["demand"]["organizationalUnits"]["teams"])
        self.team_idx = {t: i for i, t in enumerate(self.teams)}
        self.num_shifts = len(self.shift_codes)
        self.num_teams = len(self.teams)

        self.team_sizes = {
            team: sum(1 for s in self.employee_teams if team in s)
            for team in self.teams
        }

        vac_df = pd.read_csv(base / "vacations.csv", header=None)
        self.vac_mask = vac_df.iloc[:, 1:].values.astype(bool)

        dem_df = pd.read_csv(base / "demand.csv")
        dem_df["date"] = pd.to_datetime(dem_df["date"])
        start_ts = pd.Timestamp(f"{self.year}-01-01")
        dem_df["day_idx"] = (dem_df["date"] - start_ts).dt.days

        self.min_demand = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=int)
        self.ideal_demand = np.zeros((self.num_days, self.num_shifts, self.num_teams), dtype=int)
        for _, row in dem_df.iterrows():
            d = int(row["day_idx"])
            s = self.shift_idx[row["shift"]]
            t = self.team_idx[row["team"]]
            self.min_demand[d, s, t] = int(row["minimum"])
            self.ideal_demand[d, s, t] = int(row["ideal"])

        self.special_days = _build_special_days(self.year, self.num_days)
        self.special_mask = np.zeros(self.num_days, dtype=bool)
        self.special_mask[list(self.special_days)] = True
        self.member = np.array([[team in teams for team in self.teams]
                                for teams in self.employee_teams], dtype=bool)  # (E, T)

        self.max_days_per_year = 223
        self.max_consecutive_days = 5
        self.special_days_cap = 22
        self.capacity_slack = capacity_slack

        self.ideal_floor = int(max(0, self.ideal_demand.sum()
                                   - self.num_employees * self.max_days_per_year))

        self.PASS_ACTION = self.num_employees
        self.action_space = gym.spaces.Discrete(self.num_employees + 1)
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)

        self.reset()

    def build_slot_queue(self, min_demand, capacity_slack=1):
        num_days, num_shifts, num_teams = min_demand.shape
        demand_slots = []
        for d in range(num_days):
            for s in range(num_shifts):
                for t in range(num_teams):
                    headcount = int(min_demand[d, s, t])
                    for _ in range(headcount):
                        demand_slots.append((d, s, t, DEMAND))

        capacity_slots = []
        for d in range(num_days):
            for s in range(num_shifts):
                for t in range(num_teams):
                    deficit = int(self.ideal_demand[d, s, t] - min_demand[d, s, t])
                    for _ in range(deficit):
                        capacity_slots.append((d, s, t, CAPACITY))

        for _ in range(capacity_slack):
            for d in range(num_days):
                for s in range(num_shifts):
                    for t in range(num_teams):
                        capacity_slots.append((d, s, t, CAPACITY))

        return demand_slots + capacity_slots, len(demand_slots)

    def _get_assigned_shift(self, emp, day):
        s = self.emp_day_shift[emp, day]
        if s < 0:
            return None
        return self.shift_codes[s]

    def _get_prev_shift(self, emp, day):
        if day == 0:
            return None
        return self._get_assigned_shift(emp, day - 1)

    def _get_next_shift(self, emp, day):
        if day >= self.num_days - 1:
            return None
        return self._get_assigned_shift(emp, day + 1)

    def _consecutive_streak_if_work(self, emp, day):
        streak = 1
        d = day - 1
        while d >= 0 and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d -= 1
        d = day + 1
        while d < self.num_days and self.emp_day_shift[emp, d] >= 0:
            streak += 1
            d += 1
        return streak

    def _get_info(self):
        return {}

    def _obs(self):
        return np.zeros(1, dtype=np.float32)

    def current_slot(self):
        if self.slot_idx >= len(self.slot_queue):
            return None
        return self.slot_queue[self.slot_idx]

    def current_day(self):
        s = self.current_slot()
        return s[0] if s is not None else 0

    def _team_balance_bonus(self, emp_id, team):
        if not self.dual_team[emp_id]:
            return 0.0
        sizes = {t: self.team_sizes[t] for t in self.employee_teams[emp_id]}
        min_size, max_size = min(sizes.values()), max(sizes.values())
        if min_size == max_size:
            return 0.0
        smaller = {t for t, s in sizes.items() if s == min_size}
        return (max_size - min_size) * 0.5 if team in smaller else 0.0

    def _calculate_reward(self, action, day, s_idx, t_idx, kind):
        if action == self.PASS_ACTION:
            return -5.0 if kind == DEMAND else 0.0
        team = self.teams[t_idx]
        bonus = self._team_balance_bonus(action, team)
        if kind == DEMAND:
            return 1.0 + 0.2 * bonus
        cov = self.daily_coverage[day, s_idx, t_idx]
        if cov <= self.ideal_demand[day, s_idx, t_idx]:
            return 1.0 + 0.1 * bonus
        return 0.2

    def _calculate_final_reward(self):
        shortfall = int(np.maximum(0, self.min_demand - self.daily_coverage).sum())
        self.ideal_shortfall = int(np.maximum(0, self.ideal_demand - self.daily_coverage).sum())
        days_short = int(np.maximum(0, self.max_days_per_year - self.days_worked).sum())
        ideal_gap_above_min = int(np.maximum(
            0, self.ideal_demand - np.maximum(self.daily_coverage, self.min_demand)).sum())
        return -(10.0 * shortfall + 3.0 * days_short + 1.0 * ideal_gap_above_min)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.days_worked = np.zeros(self.num_employees, dtype=int)
        self.special_days_worked = np.zeros(self.num_employees, dtype=int)
        self.daily_coverage = np.zeros(
            (self.num_days, self.num_shifts, self.num_teams), dtype=np.float32
        )
        self.emp_day_shift = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.emp_day_team = np.full((self.num_employees, self.num_days), -1, dtype=int)
        self.slot_queue, self.num_demand_slots = self.build_slot_queue(self.min_demand, self.capacity_slack)
        self.cell_slots_left = np.zeros_like(self.min_demand, dtype=int)
        for d, s, t, _ in self.slot_queue:
            self.cell_slots_left[d, s, t] += 1
        self.day_slots_left = self.cell_slots_left.sum(axis=(1, 2))
        self.slot_idx = 0
        self.demand_skips = 0
        self.ideal_shortfall = 0
        self._rebuild_potential_state()
        self.phi = self._potential()
        return self._obs(), self._get_info()

    def step(self, action):
        if self.slot_idx >= len(self.slot_queue):
            return self._obs(), 0.0, True, False, self._get_info()
        day, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        action = int(action)

        if action != self.PASS_ACTION:
            self.emp_day_shift[action, day] = s_idx
            self.emp_day_team[action, day] = t_idx
            self.days_worked[action] += 1
            self.daily_coverage[day, s_idx, t_idx] += 1
            if day in self.special_days:
                self.special_days_worked[action] += 1
        elif kind == DEMAND:
            self.demand_skips += 1

        reward = self._calculate_reward(action, day, s_idx, t_idx, kind)
        self.cell_slots_left[day, s_idx, t_idx] -= 1
        self.day_slots_left[day] -= 1
        self.slot_idx += 1
        terminated = self.slot_idx >= len(self.slot_queue)
        if terminated:
            reward += self._calculate_final_reward()

        # Shaping F = Phi(s') - Phi(s), which assumes GAMMA = 1. Phi is 0 in the
        # terminal state (Grzes 2017), so the shaping sums to -Phi(s_0) per episode.
        self._update_potential_state(action, day, s_idx, t_idx)
        phi = 0.0 if terminated else self._potential()
        reward += phi - self.phi
        self.phi = phi
        return self._obs(), reward, terminated, False, self._get_info()

    def _emp_base_row(self, emp):
        # _emp_can_cover for one employee over every (day, shift), without the
        # team check: (D, S) bool.
        D = self.num_days
        shifts = self.emp_day_shift[emp]
        worked = shifts >= 0
        idx = np.arange(D)
        last_free = np.maximum.accumulate(np.where(worked, -1, idx))
        first_free = np.minimum.accumulate(np.where(worked, D, idx)[::-1])[::-1]
        left = np.zeros(D, dtype=int)
        left[1:] = idx[:-1] - last_free[:-1]
        right = np.zeros(D, dtype=int)
        right[:-1] = first_free[1:] - idx[1:]
        ok = ~worked & ~self.vac_mask[emp] & (1 + left + right <= self.max_consecutive_days)
        if self.days_worked[emp] >= self.max_days_per_year:
            ok[:] = False
        if self.special_days_worked[emp] >= self.special_days_cap:
            ok &= ~self.special_mask
        row = np.repeat(ok[:, None], self.num_shifts, axis=1)
        m, t = self.shift_idx.get("M"), self.shift_idx.get("T")
        if m is not None and t is not None:
            prev = np.concatenate(([-1], shifts[:-1]))
            nxt = np.concatenate((shifts[1:], [-1]))
            row[:, m] &= prev != t
            row[:, t] &= nxt != m
        return row

    def _rebuild_potential_state(self):
        self.base = np.stack([self._emp_base_row(e) for e in range(self.num_employees)])  # (E, D, S)
        self.n_elig = np.einsum("eds,et->dst", self.base.astype(int), self.member.astype(int))
        # Below-ideal fills each cell can still absorb in the rest of the queue.
        self.need = np.minimum(
            np.maximum(0, (self.ideal_demand - self.daily_coverage).astype(int)),
            self.cell_slots_left)
        # Days on which each employee could still be placed below ideal.
        self.placeable_day = (self.base[:, :, :, None] & self.member[:, None, None, :]
                              & (self.need > 0)[None]).any(axis=(2, 3))  # (E, D)
        self.day_avail = self.placeable_day.sum(axis=0)
        self.p_norm = self.placeable_day[:, ~self.special_mask].sum(axis=1)
        self.p_spec = self.placeable_day[:, self.special_mask].sum(axis=1)
        self.day_cap = np.zeros(self.num_days, dtype=int)
        self.day_unc_min = np.zeros(self.num_days)
        self._refresh_days(np.arange(self.num_days))

    def _refresh_days(self, days):
        n_elig, slots = self.n_elig[days], self.cell_slots_left[days]
        self.day_cap[days] = np.minimum(self.need[days], n_elig).sum(axis=(1, 2))
        min_need = np.maximum(0, self.min_demand[days] - self.daily_coverage[days])
        self.day_unc_min[days] = (np.maximum(0, min_need - np.minimum(n_elig, slots))
                                  * (slots > 0)).sum(axis=(1, 2))

    def _update_potential_state(self, action, day, s_idx, t_idx):
        was_target = self.need[day, s_idx, t_idx] > 0
        self.need[day, s_idx, t_idx] = min(
            max(0, int(self.ideal_demand[day, s_idx, t_idx] - self.daily_coverage[day, s_idx, t_idx])),
            self.cell_slots_left[day, s_idx, t_idx])
        days = np.array([day])
        if action != self.PASS_ACTION:
            row = self._emp_base_row(action)
            days = np.union1d(days, np.flatnonzero((row != self.base[action]).any(axis=1)))
            delta = row[days].astype(int) - self.base[action, days].astype(int)
            self.n_elig[days] += delta[:, :, None] * self.member[action].astype(int)
            self.base[action] = row
            new = (row[days][:, :, None] & self.member[action] & (self.need[days] > 0)).any(axis=(1, 2))
            diff = new.astype(int) - self.placeable_day[action, days].astype(int)
            self.placeable_day[action, days] = new
            self.day_avail[days] += diff
            special = self.special_mask[days]
            self.p_spec[action] += diff[special].sum()
            self.p_norm[action] += diff[~special].sum()
        if was_target != (self.need[day, s_idx, t_idx] > 0):
            new = (self.base[:, day, :, None] & self.member[:, None, :]
                   & (self.need[day] > 0)[None]).any(axis=(1, 2))
            diff = new.astype(int) - self.placeable_day[:, day].astype(int)
            self.placeable_day[:, day] = new
            self.day_avail[day] += diff.sum()
            if self.special_mask[day]:
                self.p_spec += diff
            else:
                self.p_norm += diff
        self._refresh_days(days)

    def _potential(self):
        allowance = self.max_days_per_year - self.days_worked
        special_left = np.maximum(0, self.special_days_cap - self.special_days_worked)
        u1 = np.minimum(self.day_cap, self.day_avail).sum()
        u2 = np.minimum(allowance, self.p_norm + np.minimum(self.p_spec, special_left)).sum()
        stranded = allowance.sum() - min(u1, u2)
        return -(SHAPE_W_MIN * float(self.day_unc_min.sum()) + SHAPE_W_STRANDED * float(stranded))

    def _emp_can_cover(self, emp_id, day_id, shift_id, team_id):
        if self.teams[team_id] not in self.employee_teams[emp_id]:
            return False
        if self.vac_mask[emp_id, day_id]:
            return False
        if self.emp_day_shift[emp_id, day_id] >= 0:
            return False
        if self.days_worked[emp_id] >= self.max_days_per_year:
            return False
        if day_id in self.special_days and self.special_days_worked[emp_id] >= self.special_days_cap:
            return False
        if self._consecutive_streak_if_work(emp_id, day_id) > self.max_consecutive_days:
            return False

        prev_shift = self._get_prev_shift(emp_id, day_id)
        next_shift = self._get_next_shift(emp_id, day_id)
        shift_code = self.shift_codes[shift_id]

        if shift_code == "M" and prev_shift == "T":
            return False
        if shift_code == "T" and next_shift == "M":
            return False

        return True

    def get_employee_mask(self):
        mask = np.zeros(self.num_employees + 1, dtype=bool)
        if self.slot_idx >= len(self.slot_queue):
            mask[self.PASS_ACTION] = True
            return mask
        day, s_idx, t_idx, kind = self.slot_queue[self.slot_idx]
        for e in range(self.num_employees):
            if self._emp_can_cover(e, day, s_idx, t_idx):
                mask[e] = True
        if kind == CAPACITY:
            mask[self.PASS_ACTION] = True
        elif not mask[:self.num_employees].any():
            mask[self.PASS_ACTION] = True
        return mask

    def action_label(self, emp, day):
        s = self.emp_day_shift[emp, day]
        if s < 0:
            return "-"
        t = self.emp_day_team[emp, day]
        return f"{self.shift_codes[s]}-{self.teams[t]}"

    def render(self):
        for emp in range(self.num_employees):
            schedule = [self.action_label(emp, day) for day in range(self.num_days)]
            print(f"Employee {emp + 1:2d}: {schedule}")

In [14]:
import dgl
import dgl.nn as dglnn
import torch
import numpy as np

REWARD_SCALE = 0.01


def emp_feat_dim(env):
    return 6


def day_feat_dim(env):
    return 2 * env.num_shifts + 2


def team_feat_dim(env):
    return 2 + 2 * env.num_shifts


def _static_edges(env):
    if not hasattr(env, "_static_edge_cache"):
        et_src, et_dst = [], []
        for emp in range(env.num_employees):
            for team in env.employee_teams[emp]:
                et_src.append(emp)
                et_dst.append(env.team_idx[team])

        td_src, td_dst = [], []
        for t in range(env.num_teams):
            for day in range(env.num_days):
                td_src.append(t)
                td_dst.append(day)

        env._static_edge_cache = (
            torch.tensor(et_src), torch.tensor(et_dst),
            torch.tensor(td_src), torch.tensor(td_dst),
        )
    return env._static_edge_cache


def build_graph(env):
    et_src, et_dst, td_src, td_dst = _static_edges(env)
    graph_data = {
        ("employee", "member_of", "team"): (et_src, et_dst),
        ("team", "has_member", "employee"): (et_dst, et_src),
        ("team", "demands", "day"): (td_src, td_dst),
        ("day", "demanded_by", "team"): (td_dst, td_src),
    }

    g = dgl.heterograph(graph_data, num_nodes_dict={
        "employee": env.num_employees,
        "day": env.num_days,
        "team": env.num_teams,
    })
    update_graph_features(g, env)
    return g


def get_graph(env):
    # Topology is fully static (no emp<->day edges): one graph per scenario,
    # refreshed per day via update_graph_features.
    if not hasattr(env, "_graph_cache_static"):
        env._graph_cache_static = build_graph(env)
    return env._graph_cache_static


def _static_feats(env):
    if not hasattr(env, "_static_feat_cache"):
        emp_pos = (np.arange(env.num_employees) / env.num_employees).astype(np.float32)
        special = np.array([float(d in env.special_days) for d in range(env.num_days)],
                           dtype=np.float32)
        day_pos = (np.arange(env.num_days) / env.num_days).astype(np.float32)
        team_size = np.array([env.team_sizes[t] / env.num_employees for t in env.teams],
                             dtype=np.float32)
        env._static_feat_cache = (emp_pos, special, day_pos, team_size)
    return env._static_feat_cache


def update_graph_features(g, env):
    emp_pos, special, day_pos, team_size = _static_feats(env)
    S = env.num_shifts
    D = env.num_days

    emp_feats = np.empty((env.num_employees, emp_feat_dim(env)), dtype=np.float32)
    emp_feats[:, 0] = env.days_worked / env.max_days_per_year
    cur_day = env.current_day()
    for emp in range(env.num_employees):
        emp_feats[emp, 1] = env._consecutive_streak_if_work(emp, cur_day) / env.max_consecutive_days
    emp_feats[:, 2] = emp_pos
    
    open_free = ((~env.vac_mask) & (env.emp_day_shift < 0)
                & (env.day_slots_left > 0)[None, :])
    remaining_workable = open_free.sum(axis=1)
    remaining_allowance = env.max_days_per_year - env.days_worked
    emp_feats[:, 3] = remaining_workable / env.num_days
    emp_feats[:, 4] = (remaining_workable - remaining_allowance) / env.max_days_per_year
    emp_feats[:, 5] = env.special_days_worked / env.special_days_cap

    min_gap = env.min_demand - env.daily_coverage      # (D, S, T)
    ideal_gap = env.ideal_demand - env.daily_coverage  # (D, S, T)
    day_feats = np.empty((D, day_feat_dim(env)), dtype=np.float32)
    day_feats[:, :S] = min_gap.mean(axis=2)
    day_feats[:, S:2 * S] = ideal_gap.mean(axis=2)
    day_feats[:, 2 * S] = special
    day_feats[:, 2 * S + 1] = day_pos

    # Team features: relative size + today's remaining gaps per shift
    # + special-day slack.
    team_feats = np.empty((env.num_teams, team_feat_dim(env)), dtype=np.float32)
    team_feats[:, 0] = team_size
    team_feats[:, 1:1 + S] = min_gap[cur_day].T / 10.0
    team_feats[:, 1 + S:1 + 2 * S] = ideal_gap[cur_day].T / 10.0
    # Special-day slack: Sunday/holiday shifts the team's members can still
    # work minus the special-day minimum the team still needs (cross-members
    # count for both of their teams).
    spec_open = open_free[:, env.special_mask].sum(axis=1)
    spec_left = np.minimum(env.special_days_cap - env.special_days_worked, spec_open)
    live_spec = env.special_mask & (env.day_slots_left > 0)
    spec_need = np.maximum(0, min_gap[live_spec]).sum(axis=(0, 1))
    team_feats[:, 1 + 2 * S] = (spec_left @ env.member - spec_need) / env.special_days_cap

    g.nodes["employee"].data["feat"] = torch.from_numpy(emp_feats)
    g.nodes["day"].data["feat"] = torch.from_numpy(day_feats)
    g.nodes["team"].data["feat"] = torch.from_numpy(team_feats)


def slot_gap(env, day_id, s_idx, t_idx):
    cov = env.daily_coverage[day_id, s_idx, t_idx]
    total = len(env.slot_queue)
    remaining = total - env.slot_idx
    return np.array([
        env.min_demand[day_id, s_idx, t_idx] - cov,
        env.ideal_demand[day_id, s_idx, t_idx] - cov,
        env.slot_idx / total,        # progress through the episode, in [0, 1)
        remaining * REWARD_SCALE,    # value scale, in scaled-reward units
    ], dtype=np.float32)

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GNNActorCritic(nn.Module):
    def __init__(self, emp_in_feats, day_in_feats, team_in_feats, shift_codes,
                 hidden_dim=64, encoded_dim=64):
        super().__init__()
        self.encoded_dim = encoded_dim
        self.num_shifts = len(shift_codes)

        self.emp_proj = nn.Linear(emp_in_feats, hidden_dim)
        self.day_proj = nn.Linear(day_in_feats, hidden_dim)
        self.team_proj = nn.Linear(team_in_feats, hidden_dim)

        # emp<->day messages travel through team nodes (2 hops), so the
        # convs only need the 4 static relations.
        rel_names = ["member_of", "has_member", "demands", "demanded_by"]

        self.conv1 = dglnn.HeteroGraphConv({
            rel: dglnn.SAGEConv(hidden_dim, hidden_dim, 'mean') for rel in rel_names
        }, aggregate='sum')

        self.conv2 = dglnn.HeteroGraphConv({
            rel: dglnn.SAGEConv(hidden_dim, encoded_dim, 'mean') for rel in rel_names
        }, aggregate='sum')

        ctx_dim = encoded_dim + encoded_dim + 2 + self.num_shifts + 1  # 64+64+2+2+1 = 133
        head_in = encoded_dim + ctx_dim

        # Pointer score: one logit per employee
        self.score_head  = nn.Sequential(
            nn.Linear(head_in, 64), 
            nn.Tanh(), 
            nn.Linear(64, 1)
        )

        # PASS + value condition on a workforce summary (mean emp_emb) + slot ctx.
        self.pass_head   = nn.Sequential(
            nn.Linear(head_in, 64),
            nn.Tanh(), 
            nn.Linear(64, 1)
        )

        self.critic_head = nn.Sequential(
            nn.Linear(head_in + 1, 64),
            nn.Tanh(),
            nn.Linear(64, 2)
        )

        nn.init.orthogonal_(self.critic_head[-1].weight, gain=1.0)
        with torch.no_grad():
            self.critic_head[-1].bias.copy_(torch.tensor([0.5, 0.0]))  # slope, intercept

    @classmethod
    def from_env(cls, env, hidden_dim=64, encoded_dim=64):
        return cls(
            emp_in_feats=emp_feat_dim(env),
            day_in_feats=day_feat_dim(env),
            team_in_feats=team_feat_dim(env),
            shift_codes=env.shift_codes,
            hidden_dim=hidden_dim, 
            encoded_dim=encoded_dim,
        )

    def gnn_forward(self, g):
        h = {
            "employee": self.emp_proj(g.nodes["employee"].data["feat"]),
            "day": self.day_proj(g.nodes["day"].data["feat"]),
            "team": self.team_proj(g.nodes["team"].data["feat"]),
        }

        h = self.conv1(g, h); 
        h = {k: F.relu(v) for k, v in h.items()}
        h = self.conv2(g, h); 
        h = {k: F.relu(v) for k, v in h.items()}
        return h["employee"], h["day"], h["team"]

    def _heads_multi(self, emp_emb_b, day_vec, team_vec, slot_gaps, shift, kind, action_masks):
        B, E, _ = emp_emb_b.shape
        gaps = slot_gaps[:, :2]
        progress = slot_gaps[:, 2:3]
        value_scale = slot_gaps[:, 3]
        ctx = torch.cat([day_vec, team_vec, gaps / 10.0, shift, kind], dim=-1)  # (B, 133)

        ctx_b = ctx.unsqueeze(1).expand(B, E, -1)
        emp_logits = self.score_head(torch.cat([emp_emb_b, ctx_b], dim=-1)).squeeze(-1)  # (B, E)

        pooled = torch.cat([emp_emb_b.mean(1), ctx], dim=-1)
        pass_logit = self.pass_head(pooled)

        ab = self.critic_head(torch.cat([pooled, progress], dim=-1))
        values = ab[:, 0] * value_scale + ab[:, 1]

        logits = torch.cat([emp_logits, pass_logit], dim=-1)                      # (B, E+1)
        masks_bool = torch.as_tensor(action_masks, dtype=torch.bool)
        if masks_bool.dim() == 1:
            masks_bool = masks_bool.unsqueeze(0)
        all_invalid = ~masks_bool.any(dim=-1)
        if all_invalid.any():
            masks_bool = masks_bool.clone()
            masks_bool[all_invalid, -1] = True
        logits = logits.masked_fill(~masks_bool, float("-inf"))
        return F.softmax(logits, dim=-1), values

    def _heads(self, emp_emb, day_emb, team_emb, day_ids, team_ids,
               slot_gaps, shift, kind, action_masks):
        # Single-snapshot case: the whole batch shares one embedding set.
        B = day_ids.shape[0]
        emp_emb_b = emp_emb.unsqueeze(0).expand(B, -1, -1)
        return self._heads_multi(emp_emb_b, day_emb[day_ids], team_emb[team_ids],
                                 slot_gaps, shift, kind, action_masks)

    def forward(self, g, day_ids, team_ids, slot_gaps, shift, kind, action_masks):
        emp_emb, day_emb, team_emb = self.gnn_forward(g)
        return self._heads(emp_emb, day_emb, team_emb, day_ids, team_ids,
                           slot_gaps, shift, kind, action_masks)

In [16]:
from dataclasses import dataclass, field
from torch.distributions import Categorical


@dataclass
class Trajectory:
    day_ids: list = field(default_factory=list)
    team_ids: list = field(default_factory=list)
    slot_gaps: list = field(default_factory=list)
    shifts: list = field(default_factory=list)
    kinds: list = field(default_factory=list)
    action_masks: list = field(default_factory=list)
    actions: list = field(default_factory=list)
    log_probs_old: list = field(default_factory=list)
    rewards: list = field(default_factory=list)
    values: list = field(default_factory=list)
    snap_ids: list = field(default_factory=list)
    emp_feats_unique: list = field(default_factory=list)   # one (E, F_emp) tensor per snapshot
    day_feats_unique: list = field(default_factory=list)   # one (D, F_day) tensor per snapshot
    team_feats_unique: list = field(default_factory=list)  # one (T, F_team) tensor per snapshot


    def to_tensors(self):
        return {
            "day_ids": torch.tensor(self.day_ids, dtype=torch.long),
            "team_ids": torch.tensor(self.team_ids, dtype=torch.long),
            "slot_gaps": torch.stack(self.slot_gaps),
            "shifts": torch.stack(self.shifts),
            "kinds": torch.stack(self.kinds),
            "action_masks": torch.stack(self.action_masks),
            "actions": torch.tensor(self.actions, dtype=torch.long),
            "log_probs_old": torch.tensor(self.log_probs_old, dtype=torch.float32),
            "rewards": torch.tensor(self.rewards, dtype=torch.float32),
            "values": torch.tensor(self.values, dtype=torch.float32),
            "snap_ids": torch.tensor(self.snap_ids, dtype=torch.long),
            "emp_feats_unique": torch.stack(self.emp_feats_unique),
            "day_feats_unique": torch.stack(self.day_feats_unique),
            "team_feats_unique": torch.stack(self.team_feats_unique),
        }


def collect_trajectory(env, model):
    model.eval()
    traj = Trajectory()
    env.reset()
    terminated = truncated = False

    graph = get_graph(env)
    cached_day_id = None
    cached_emp_emb = None
    cached_day_emb = None
    cached_team_emb = None


    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id, s_idx, t_idx, kind = slot

            if day_id != cached_day_id:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb, cached_team_emb = model.gnn_forward(graph)
                cached_day_id = day_id
                traj.emp_feats_unique.append(graph.nodes["employee"].data["feat"].clone())
                traj.day_feats_unique.append(graph.nodes["day"].data["feat"].clone())
                traj.team_feats_unique.append(graph.nodes["team"].data["feat"].clone())
                cur_snap = len(traj.emp_feats_unique) - 1

            # Per-step inputs describing THIS slot.
            gap = slot_gap(env, day_id, s_idx, t_idx)
            shift_oh = [0.0] * env.num_shifts; shift_oh[s_idx] = 1.0
            mask = env.get_employee_mask()

            day_id_t = torch.tensor([day_id], dtype=torch.long)
            team_id_t = torch.tensor([t_idx], dtype=torch.long)
            gap_t = torch.from_numpy(gap).unsqueeze(0)
            shift_t = torch.tensor([shift_oh], dtype=torch.float32)
            kind_t = torch.tensor([[float(kind)]], dtype=torch.float32)
            mask_t = torch.tensor(mask.tolist(), dtype=torch.bool).unsqueeze(0)

            probs, values = model._heads(
                cached_emp_emb, cached_day_emb, cached_team_emb,
                day_id_t, team_id_t, gap_t, shift_t, kind_t, mask_t,
            )

            dist = Categorical(probs=probs[0])
            action = dist.sample()

            _, reward, terminated, truncated, _ = env.step(action.item())

            traj.day_ids.append(day_id)
            traj.team_ids.append(t_idx)
            traj.slot_gaps.append(gap_t[0])
            traj.shifts.append(shift_t[0])
            traj.kinds.append(kind_t[0])
            traj.action_masks.append(mask_t[0])
            traj.actions.append(action.item())
            traj.log_probs_old.append(dist.log_prob(action).item())
            traj.rewards.append(float(reward))
            traj.values.append(values[0].item())
            traj.snap_ids.append(cur_snap)

    return traj

In [17]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    T = len(rewards)
    rewards_a = np.array(rewards, dtype=np.float32)
    values_a = np.array(values,  dtype=np.float32)
    values_ext = np.append(values_a, 0.0)

    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(T)):
        delta = rewards_a[t] + gamma * values_ext[t + 1] - values_ext[t]
        gae = delta + gamma * lam * gae
        advantages[t] = gae

    returns = advantages + values_a
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-5)

    return (
        torch.tensor(advantages, dtype=torch.float32),
        torch.tensor(returns, dtype=torch.float32),
    )


In [18]:
from torch.distributions import Categorical


def ppo_update(model, optimizer, batch, advantages, returns, graph,
               clip_eps=0.2, value_coeff=0.5, entropy_coeff=0.01,
               K_epochs=4, mini_batch_size=512, target_kl=None):
    T = batch["actions"].shape[0]
    snap_ids = batch["snap_ids"]
    U = int(snap_ids.max().item()) + 1
    E = graph.num_nodes("employee")
    D = graph.num_nodes("day")
    TM = graph.num_nodes("team")
    losses, track_entropy, track_kl = [], [], []

    model.train()

    for _ in range(K_epochs):
        # Shuffle whole snapshot groups (keeps each day's steps together so a
        # minibatch touches only a handful of snapshots).
        rank = torch.randperm(U)
        order = torch.argsort(rank[snap_ids], stable=True)
        epoch_kls = []

        for start in range(0, T, mini_batch_size):
            idx = order[start:start + mini_batch_size]
            snaps, local = torch.unique(snap_ids[idx], return_inverse=True)
            n = snaps.shape[0]

            # Topology is static: replicate the one scenario graph n times and
            # load each snapshot's stored node features into its component.
            bg = dgl.batch([graph] * n)
            bg.nodes["employee"].data["feat"] = batch["emp_feats_unique"][snaps].reshape(n * E, -1)
            bg.nodes["day"].data["feat"] = batch["day_feats_unique"][snaps].reshape(n * D, -1)
            bg.nodes["team"].data["feat"] = batch["team_feats_unique"][snaps].reshape(n * TM, -1)
            emp_emb, day_emb, team_emb = model.gnn_forward(bg)
            emp_emb = emp_emb.view(n, E, -1)
            day_emb = day_emb.view(n, D, -1)
            team_emb = team_emb.view(n, TM, -1)

            probs, values = model._heads_multi(
                emp_emb[local],
                day_emb[local, batch["day_ids"][idx]],
                team_emb[local, batch["team_ids"][idx]],
                batch["slot_gaps"][idx],
                batch["shifts"][idx], batch["kinds"][idx],
                batch["action_masks"][idx],
            )

            dist = Categorical(probs=probs)
            logp = dist.log_prob(batch["actions"][idx])
            ratios = torch.exp(logp - batch["log_probs_old"][idx])
            with torch.no_grad():
                approx_kl = ((ratios - 1) - torch.log(ratios)).mean().item()
            epoch_kls.append(approx_kl)
            adv = advantages[idx]
            surr1 = ratios * adv
            surr2 = torch.clamp(ratios, 1 - clip_eps, 1 + clip_eps) * adv
            actor_loss = -torch.min(surr1, surr2).mean() - entropy_coeff * dist.entropy().mean()
            critic_loss = F.smooth_l1_loss(values, returns[idx])
            loss = actor_loss + value_coeff * critic_loss

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            losses.append(loss.item())
            track_entropy.append(dist.entropy().mean().item())

        track_kl.extend(epoch_kls)
        if target_kl is not None and float(np.mean(epoch_kls)) > target_kl:
            break

    return (float(np.mean(losses)), float(np.mean(track_entropy)),
            float(np.mean(track_kl)))


In [19]:
PROBLEMS_ROOT = "../../../../data/problems"
POOL = ["SMARTASK_SIMPLE_2025_CASE1", "SMARTASK_SIMPLE_2025_CASE4",
        "SMARTASK_4TEAMS_24EMP_CASE1", "SMARTASK_4TEAMS_24EMP_CASE4"]

HOLDOUT = "SMARTASK_8TEAMS_48EMP"

_scenario_cache = {}

def get_scenario(name):
    if name not in _scenario_cache:
        _scenario_cache[name] = ScheduleEnv(data_dir=f"{PROBLEMS_ROOT}/{name}")
    return _scenario_cache[name]


def evaluate(model, env, greedy=False):
    model.eval()
    env.reset()
    terminated = truncated = False
    total_reward = 0.0

    graph = get_graph(env)
    cached_day_id = None
    cached_emp_emb = None
    cached_day_emb = None
    cached_team_emb = None

    with torch.no_grad():
        while not (terminated or truncated):
            slot = env.current_slot()
            if slot is None:
                break
            day_id, s_idx, t_idx, kind = slot

            if day_id != cached_day_id:
                update_graph_features(graph, env)
                cached_emp_emb, cached_day_emb, cached_team_emb = model.gnn_forward(graph)
                cached_day_id = day_id

            gap = slot_gap(env, day_id, s_idx, t_idx)
            shift_oh = [0.0] * env.num_shifts; shift_oh[s_idx] = 1.0
            mask = env.get_employee_mask()

            day_id_t = torch.tensor([day_id], dtype=torch.long)
            team_id_t = torch.tensor([t_idx], dtype=torch.long)
            gap_t = torch.from_numpy(gap).unsqueeze(0)
            shift_t = torch.tensor([shift_oh], dtype=torch.float32)
            kind_t = torch.tensor([[float(kind)]], dtype=torch.float32)
            mask_t = torch.tensor(mask.tolist(), dtype=torch.bool).unsqueeze(0)

            probs, _ = model._heads(
                cached_emp_emb, cached_day_emb, cached_team_emb,
                day_id_t, team_id_t, gap_t, shift_t, kind_t, mask_t,
            )

            action = probs[0].argmax() if greedy else Categorical(probs=probs[0]).sample()
            _, reward, terminated, truncated, _ = env.step(action.item())
            total_reward += reward

    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    ideal_gap = int(np.maximum(0, env.ideal_demand - env.daily_coverage).sum())

    snapshot = {
        "demand_skips": env.demand_skips,
        "ideal_gap": ideal_gap,
        "daily_coverage": env.daily_coverage.copy(),
    }

    schedule = [
        [env.action_label(emp, day) for day in range(env.num_days)]
        for emp in range(env.num_employees)
    ]

    return total_reward, shortfall, env.days_worked.tolist(), schedule, snapshot

In [20]:
import os

NUM_EPISODES = 10000
GAMMA = 1.0   # finite-horizon episodic task: terminal reward must reach every decision undiscounted
LAM = 0.95
CLIP_EPS = 0.15
VALUE_COEFF = 0.5
ENTROPY_COEFF_0 = 0.01
ENTROPY_DECAY = 0.995   # v2 schedule (per episode -> each scenario sees ~0.995^2)
ENTROPY_MIN = 0.003     # v2 floor
TARGET_KL = None
K_EPOCHS = 3
MINI_BATCH_SIZE = 512
LR = 1.5e-4
SAVE_EVERY = 20
EVAL_SAMPLES = 3     # sampled rollouts per pool scenario for checkpoint selection
HOLDOUT_EVERY = 100  # episodes, log-only (see below)
HOLDOUT_SAMPLES = 2

W_SHORTFALL = 10.0   # hard constraint: minimum demand not met
W_DAYS_SHORT = 3.0   # hard constraint: contract days not reached
W_EXCESS = 1.0       # soft objective: coverage below ideal

BEST_CKPT = "best_multiteam_v4g_c14.pth"
LATEST_CKPT = "latest_multiteam_v4g_c14.pth"
RESUME_FROM = LATEST_CKPT if os.path.exists(LATEST_CKPT) else None

model = GNNActorCritic.from_env(get_scenario(POOL[0]))
optimizer = torch.optim.Adam(model.parameters(), lr=LR, eps=1e-5)

INF_METRIC = float("inf")
best_metric = INF_METRIC
start_episode = 0

if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    for group in optimizer.param_groups:
        group["lr"] = LR
    start_episode = ckpt["episode"]
    bm = ckpt.get("best_metric", INF_METRIC)
    best_metric = float(bm) if isinstance(bm, (int, float)) else INF_METRIC
    print(f"Resumed from episode {start_episode}, best metric = {best_metric}")


def eval_scenario(model, env):
    best = None
    for k in range(EVAL_SAMPLES):
        torch.manual_seed(1234 + k)
        _, sf, days_worked, _, snap = evaluate(model, env, greedy=False)
        ds = int(np.maximum(0, env.max_days_per_year - np.array(days_worked)).sum())
        ex = max(0, snap["ideal_gap"] - env.ideal_floor)
        if best is None or (sf, ds, ex) < best:
            best = (sf, ds, ex)
    return best


for episode in range(start_episode, NUM_EPISODES):
    entropy_coeff = max(ENTROPY_MIN, ENTROPY_COEFF_0 * (ENTROPY_DECAY ** episode))

    # Round-robin over the pool: deterministic interleave keeps gradient
    # exposure balanced between scenarios.
    name = POOL[episode % len(POOL)]
    env = get_scenario(name)

    traj = collect_trajectory(env, model)
    batch = traj.to_tensors()

    scaled_rewards = [r * REWARD_SCALE for r in traj.rewards]
    advantages, returns = compute_gae(scaled_rewards, traj.values, GAMMA, LAM)

    mean_loss, mean_entropy, mean_kl = ppo_update(
        model, optimizer, batch, advantages, returns, get_graph(env),
        clip_eps=CLIP_EPS,
        value_coeff=VALUE_COEFF,
        entropy_coeff=entropy_coeff,
        K_epochs=K_EPOCHS,
        mini_batch_size=MINI_BATCH_SIZE,
        target_kl=TARGET_KL,
    )

    total_reward = sum(traj.rewards)
    shortfall = int(np.maximum(0, env.min_demand - env.daily_coverage).sum())
    ideal_gap = int(np.maximum(0, env.ideal_demand - env.daily_coverage).sum())
    days_short = int(np.maximum(0, env.max_days_per_year - env.days_worked).sum())

    # % 5 with a 2-scenario round-robin alternates which scenario gets logged.
    if episode % 5 == 0:
        print(f"Ep {episode:>4} [{name}] | R={total_reward:>8.1f} "
              f"| Loss={mean_loss:.4f} | ent={mean_entropy:.4f} (coeff={entropy_coeff:.4f}) "
              f"| kl={mean_kl:.4f} "
              f"| shortfall={shortfall} | ideal_excess={max(0, ideal_gap - env.ideal_floor)} "
              f"| skips={env.demand_skips} | days_short={days_short} "
              f"| days={env.days_worked.mean():.0f}")

    if (episode + 1) % SAVE_EVERY == 0:
        rng_state = torch.get_rng_state()
        metric_sf, metric_ds, metric_ex = 0.0, 0.0, 0.0
        parts = []
        for eval_name in POOL:
            eval_env = get_scenario(eval_name)
            sf, ds, ex = eval_scenario(model, eval_env)
            total_min = eval_env.min_demand.sum()
            metric_sf += sf / total_min
            metric_ds += ds / (eval_env.num_employees * eval_env.max_days_per_year)
            metric_ex += ex / total_min
            parts.append(f"{eval_name}: shortfall={sf} ({100 * (1 - sf / total_min):.2f}% cov) "
                        f"days_short={ds} excess={ex}")
        torch.set_rng_state(rng_state)
        # float(): the fractions come out as numpy scalars, which
        # weights_only=True loading refuses to unpickle.
        metric = float(W_SHORTFALL * metric_sf
                       + W_DAYS_SHORT * metric_ds
                       + W_EXCESS * metric_ex)

        print(f"  eval (best of {EVAL_SAMPLES}) @ ep {episode + 1}: " + " | ".join(parts)
              + f" || metric={metric:.4f}")
        if metric < best_metric:
            best_metric = metric
            torch.save(
                {"episode": episode + 1, "model_state_dict": model.state_dict(),
                 "optimizer_state_dict": optimizer.state_dict(), "best_metric": best_metric},
                BEST_CKPT,
            )
            torch.save(
                {"episode": episode + 1, "model_state_dict": model.state_dict(),
                 "optimizer_state_dict": optimizer.state_dict(), "best_metric": best_metric},
                BEST_CKPT.replace(".pth", f"_ep{episode + 1}.pth")
            )
            print(f"  NEW BEST: metric = {metric:.4f} "
                  f"(shortfall {metric_sf:.4f} | days_short {metric_ds:.4f} "
                  f"| excess {metric_ex:.4f})")

        # LATEST is saved after the eval so a resume picks up the current
        # best_metric.
        torch.save(
            {"episode": episode + 1, "model_state_dict": model.state_dict(),
             "optimizer_state_dict": optimizer.state_dict(), "best_metric": best_metric},
            LATEST_CKPT,
        )

    # Zero-shot monitoring on the holdout. LOG-ONLY: must never touch
    # best_metric or checkpoint selection.
    if (episode + 1) % HOLDOUT_EVERY == 0:
        rng_state = torch.get_rng_state()
        h_env = get_scenario(HOLDOUT)
        h_best = None
        for k in range(HOLDOUT_SAMPLES):
            torch.manual_seed(1234 + k)
            _, sf, _, _, snap = evaluate(model, h_env, greedy=False)
            ex = max(0, snap["ideal_gap"] - h_env.ideal_floor)
            if h_best is None or (sf, ex) < h_best:
                h_best = (sf, ex)
        torch.set_rng_state(rng_state)
        print(f"  [holdout {HOLDOUT}] @ ep {episode + 1}: "
              f"shortfall={h_best[0]} | ideal_excess={h_best[1]} (log-only)")

Ep    0 [SMARTASK_SIMPLE_2025_CASE1] | R=  1722.3 | Loss=0.4600 | ent=1.0048 (coeff=0.0100) | kl=0.0002 | shortfall=42 | ideal_excess=242 | skips=42 | days_short=83 | days=216
Ep    5 [SMARTASK_SIMPLE_2025_CASE4] | R=  1663.0 | Loss=0.0610 | ent=0.9666 (coeff=0.0098) | kl=0.0003 | shortfall=83 | ideal_excess=258 | skips=83 | days_short=7 | days=222
Ep   10 [SMARTASK_4TEAMS_24EMP_CASE1] | R=  4199.2 | Loss=0.1080 | ent=0.8180 (coeff=0.0095) | kl=0.0003 | shortfall=3 | ideal_excess=346 | skips=3 | days_short=0 | days=223
Ep   15 [SMARTASK_4TEAMS_24EMP_CASE4] | R=  4162.2 | Loss=0.0850 | ent=0.8023 (coeff=0.0093) | kl=0.0001 | shortfall=5 | ideal_excess=351 | skips=5 | days_short=0 | days=223
  eval (best of 3) @ ep 20: SMARTASK_SIMPLE_2025_CASE1: shortfall=36 (98.27% cov) days_short=0 excess=195 | SMARTASK_SIMPLE_2025_CASE4: shortfall=75 (96.40% cov) days_short=0 excess=250 | SMARTASK_4TEAMS_24EMP_CASE1: shortfall=2 (99.93% cov) days_short=0 excess=252 | SMARTASK_4TEAMS_24EMP_CASE4: shor

KeyboardInterrupt: 

In [21]:
import csv
import time

ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
bm = ckpt["best_metric"]
bm = bm if isinstance(bm, float) else tuple(bm)
ckpt_step = ckpt.get("iteration", ckpt.get("episode"))
print(f"Checkpoint: step {ckpt_step}, best metric = {bm}\n")

# Full sweep: every scenario size x every vacation map. The _CASE<n> scenario
# folders are built by scripts/generate_problem_scenarios.py, which copies
# api/src/main/resources/vacationData/<E>_employees/templates/
# VacationTemplate_Case<n>_<E>.csv into each folder as vacations.csv.
# _CASE1 is byte-identical to the old unsuffixed folders, so Case 1 numbers
# stay directly comparable to earlier runs. SIMPLE_2025 is the exception:
# its CASE1 keeps SIMPLE's original vacations.csv (the map trained on), and
# only CASE2-4 come from the 12-employee templates.
EVAL_SCENARIOS = ["SMARTASK_SIMPLE_2025", "SMARTASK_4TEAMS_24EMP",
                  "SMARTASK_8TEAMS_48EMP", "SMARTASK_16TEAMS_96EMP"]
EVAL_CASES = (1, 2, 3, 4)
N = 10

rows = []
for base_name in EVAL_SCENARIOS:
    for case in EVAL_CASES:
        name = f"{base_name}_CASE{case}"
        env_e = get_scenario(name)
        total_min = env_e.min_demand.sum()
        total_ideal = env_e.ideal_demand.sum()

        results = []
        t0 = time.perf_counter()
        for i in range(N):
            torch.manual_seed(42 + i)  # same seeds as the single-scenario eval
            results.append(evaluate(model, env_e, greedy=False))
        time_taken = time.perf_counter() - t0

        # Raw ideal gap (ideal demand left uncovered), no floor subtracted.
        sfs = [r[1] for r in results]
        gaps = [r[4]["ideal_gap"] for r in results]

        # Mean over the fixed-seed runs is the honest statistic; best-of-N is
        # the deployment strategy (generate N schedules, keep the best).
        best_i = min(range(N), key=lambda i: (sfs[i], gaps[i]))
        best_sf, best_gap = sfs[best_i], gaps[best_i]
        best_schedule = results[best_i][3]

        # The checkpoint was trained on POOL under Case 1 only; every other
        # (scenario, case) pair is zero-shot to some degree.
        seen = "train" if (base_name in POOL and case == 1) else "zero-shot"

        rows.append({
            "scenario": base_name.replace("SMARTASK_", ""),
            "case": case,
            "seen": seen,
            "Avg Shortfall": float(np.mean(sfs)),
            "Best Shortfall": best_sf,
            "Avg Ideal Gap": float(np.mean(gaps)),
            "Best Ideal Gap": best_gap,
            "Time Taken (s)": time_taken,
            "% Min. Not Filled": 100 * best_sf / total_min,
            "% Ideals Not Filled": 100 * best_gap / total_ideal,
        })

        schedule_csv = f"best_schedule_{base_name}_case{case}_multiteam_v4.csv"
        with open(schedule_csv, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Employee"]
                            + [f"Day_{d}" for d in range(len(best_schedule[0]))])
            for emp in range(len(best_schedule)):
                writer.writerow([f"Employee_{emp + 1}"] + best_schedule[emp])

        print(f"[done] {name:<32} avg_sf={np.mean(sfs):>7.1f} best_sf={best_sf:<6} "
              f"best_gap={best_gap:<7} time={time_taken:.1f}s -> {schedule_csv}")

df = pd.DataFrame(rows)
print("\n" + "=" * 130)
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print("=" * 130)

summary_csv = "eval_summary_vacation_cases_multiteam_v4_c14.csv"
df.to_csv(summary_csv, index=False)
print(f"\nSummary written to {summary_csv}")


Checkpoint: step 420, best metric = 0.7064326429293791

[done] SMARTASK_SIMPLE_2025_CASE1       avg_sf=   19.9 best_sf=18     best_gap=151     time=68.2s -> best_schedule_SMARTASK_SIMPLE_2025_case1_multiteam_v4.csv
[done] SMARTASK_SIMPLE_2025_CASE2       avg_sf=   19.6 best_sf=17     best_gap=170     time=65.3s -> best_schedule_SMARTASK_SIMPLE_2025_case2_multiteam_v4.csv
[done] SMARTASK_SIMPLE_2025_CASE3       avg_sf=   20.7 best_sf=20     best_gap=155     time=65.6s -> best_schedule_SMARTASK_SIMPLE_2025_case3_multiteam_v4.csv
[done] SMARTASK_SIMPLE_2025_CASE4       avg_sf=   60.7 best_sf=57     best_gap=215     time=65.1s -> best_schedule_SMARTASK_SIMPLE_2025_case4_multiteam_v4.csv
[done] SMARTASK_4TEAMS_24EMP_CASE1      avg_sf=    3.1 best_sf=1      best_gap=674     time=110.2s -> best_schedule_SMARTASK_4TEAMS_24EMP_case1_multiteam_v4.csv
[done] SMARTASK_4TEAMS_24EMP_CASE2      avg_sf=    2.2 best_sf=0      best_gap=653     time=110.4s -> best_schedule_SMARTASK_4TEAMS_24EMP_case2_mul

In [ ]:
import matplotlib.pyplot as plt

PLOT_SCENARIO = HOLDOUT   # any name from POOL, or HOLDOUT for the zero-shot scenario
env_p = get_scenario(PLOT_SCENARIO)

ckpt = torch.load(BEST_CKPT, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
_, _, days_worked_list, _, snap = evaluate(model, env_p, greedy=False)

days   = np.arange(env_p.num_days)
cov    = snap["daily_coverage"]   # (num_days, num_shifts, num_teams)
demand = env_p.min_demand         # (num_days, num_shifts, num_teams)

# Coverage-vs-demand grid: one subplot per (shift, team) pair.
fig, axes = plt.subplots(
    env_p.num_shifts, env_p.num_teams,
    figsize=(4 * env_p.num_teams, 4 * env_p.num_shifts),
    sharex=True, sharey=True, squeeze=False,
)
for s_idx, shift in enumerate(env_p.shift_codes):
    for t_idx, team in enumerate(env_p.teams):
        ax = axes[s_idx][t_idx]
        ax.plot(days, cov[:, s_idx, t_idx], label='Coverage', alpha=0.7)
        ax.plot(days, demand[:, s_idx, t_idx], label='Demand', alpha=0.7, linestyle='--', color='r')
        ax.fill_between(
            days, cov[:, s_idx, t_idx], demand[:, s_idx, t_idx],
            where=cov[:, s_idx, t_idx] < demand[:, s_idx, t_idx],
            color='red', alpha=0.2, label='Shortfall',
        )
        ax.set_title(f"{shift}-{team}")
        if t_idx == 0:
            ax.set_ylabel('Employees')
        if s_idx == env_p.num_shifts - 1:
            ax.set_xlabel('Day of year')
        ax.legend(loc='upper right', fontsize=8)

fig.suptitle(f'Coverage vs Demand across the year — {PLOT_SCENARIO}', fontsize=14)
plt.tight_layout()
plt.show()

# Quarterly violation breakdown — one column per (shift, team) pair plus a total.
q_edges = np.linspace(0, env_p.num_days, 5, dtype=int)
quarters = list(zip(q_edges[:-1], q_edges[1:]))
pair_labels = [f"{s}-{t}" for t in env_p.teams for s in env_p.shift_codes]
header = f"{'Quarter':<12} " + " ".join(f"{p:>8}" for p in pair_labels) + f" {'Total':>6}"
print()
print(header)
print("-" * len(header))
total_viol = 0
for q, (start, end) in enumerate(quarters, 1):
    per_pair = []
    q_total = 0
    for team in env_p.teams:
        for shift in env_p.shift_codes:
            s_idx, t_idx = env_p.shift_idx[shift], env_p.team_idx[team]
            v = int(np.sum(np.maximum(0, demand[start:end, s_idx, t_idx] - cov[start:end, s_idx, t_idx])))
            per_pair.append(v)
            q_total += v
    total_viol += q_total
    print(f"Q{q} ({start:3d}-{end:3d})  " + " ".join(f"{v:>8}" for v in per_pair) + f" {q_total:>6}")
print("-" * len(header))
print(f"{'Total':<12} " + " ".join(f"{'':>8}" for _ in pair_labels) + f" {total_viol:>6}")

days_worked_arr = np.array(days_worked_list)
print(f"\nDays worked/employee: {days_worked_list}")
print(f"Mean: {days_worked_arr.mean():.1f}, Min: {days_worked_arr.min()}, Max: {days_worked_arr.max()}")
